# Project 77 — Preference Pair Builder
## Generate Chosen/Rejected Pairs for RLHF-Style Training

**Stack:** LangChain · Ollama · Pydantic · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic pandas

## Step 1 — Define Tasks & Generate Candidates

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import json, pandas as pd
from pathlib import Path

judge = ChatOllama(model="qwen3:8b", temperature=0.0)

prompts = [
    "Write a product description for wireless earbuds",
    "Explain recursion to a beginner",
    "Draft a LinkedIn post about starting a new job",
    "Write a bug report for a login failure",
    "Create a meeting agenda for a sprint retrospective",
    "Explain why unit tests are important",
]

# Generate 3 candidates per prompt at different temperatures
all_candidates = {}
for prompt in prompts:
    outputs = []
    for temp in [0.1, 0.5, 0.9]:
        gen = ChatOllama(model="qwen3:8b", temperature=temp)
        chain = ChatPromptTemplate.from_template("{p}") | gen | StrOutputParser()
        outputs.append(chain.invoke({"p": prompt}))
    all_candidates[prompt] = outputs
    print(f"  Generated 3 candidates for: {prompt[:40]}...")

print(f"\nTotal: {sum(len(v) for v in all_candidates.values())} candidate outputs")

## Step 2 — Pairwise Ranking

In [ ]:
class PairwiseJudgment(BaseModel):
    winner: str = Field(description="A or B")
    score_a: int = Field(ge=1, le=5)
    score_b: int = Field(ge=1, le=5)
    reasoning: str
    criteria_used: list[str]

ranker = judge.with_structured_output(PairwiseJudgment)

preference_pairs = []
for prompt, outputs in all_candidates.items():
    # Compare all pairs
    best_idx, worst_idx = 0, 0
    best_score, worst_score = 0, 10

    for i in range(len(outputs)):
        for j in range(i+1, len(outputs)):
            judgment = ranker.invoke(
                f"Prompt: {prompt}\n\n"
                f"Response A:\n{outputs[i][:400]}\n\n"
                f"Response B:\n{outputs[j][:400]}\n\n"
                f"Which response is better? Evaluate: relevance, clarity, completeness."
            )
            if judgment.winner == "A" and judgment.score_a > best_score:
                best_idx, best_score = i, judgment.score_a
            elif judgment.winner == "B" and judgment.score_b > best_score:
                best_idx, best_score = j, judgment.score_b

    # Set worst as the other extreme
    scores = []
    for i in range(len(outputs)):
        if i != best_idx:
            j_check = ranker.invoke(
                f"Rate this response to '{prompt[:40]}...':\n{outputs[i][:300]}\n\n"
                f"Score 1-5:"
            )
            scores.append((i, j_check.score_a))
    worst_idx = min(scores, key=lambda x: x[1])[0] if scores else (1 if best_idx == 0 else 0)

    preference_pairs.append({
        "prompt": prompt,
        "chosen": outputs[best_idx][:500],
        "rejected": outputs[worst_idx][:500],
        "best_temp": [0.1, 0.5, 0.9][best_idx],
        "worst_temp": [0.1, 0.5, 0.9][worst_idx],
    })
    print(f"  {prompt[:40]}... best=T{[0.1,0.5,0.9][best_idx]}, worst=T{[0.1,0.5,0.9][worst_idx]}")

## Step 3 — Export in DPO Format

In [ ]:
# DPO format
dpo_data = [{"prompt": p["prompt"], "chosen": p["chosen"], "rejected": p["rejected"]}
            for p in preference_pairs]

Path("sample_data").mkdir(exist_ok=True)
with open("sample_data/preference_pairs.json", "w") as f:
    json.dump(dpo_data, f, indent=2)

# RLHF format
rlhf_data = []
for p in preference_pairs:
    rlhf_data.append({"prompt": p["prompt"], "response": p["chosen"], "label": 1})
    rlhf_data.append({"prompt": p["prompt"], "response": p["rejected"], "label": 0})

with open("sample_data/rlhf_pairs.jsonl", "w") as f:
    for item in rlhf_data:
        f.write(json.dumps(item) + "\n")

print(f"Exported {len(dpo_data)} DPO pairs → sample_data/preference_pairs.json")
print(f"Exported {len(rlhf_data)} RLHF entries → sample_data/rlhf_pairs.jsonl")

## Step 4 — Temperature Analysis

In [ ]:
temp_results = pd.DataFrame([{
    "prompt": p["prompt"][:30],
    "best_temp": p["best_temp"],
    "worst_temp": p["worst_temp"],
} for p in preference_pairs])

print("TEMPERATURE PREFERENCE ANALYSIS")
print("=" * 50)
print(f"Best temperature distribution: {temp_results['best_temp'].value_counts().to_dict()}")
print(f"Worst temperature distribution: {temp_results['worst_temp'].value_counts().to_dict()}")
print(f"\nInsight: Temperature {temp_results['best_temp'].mode().iloc[0]} most often produces best results")

## What You Learned
- **Pairwise preference ranking** for RLHF
- **Multi-temperature candidate generation**
- **DPO & RLHF format export**
- **Temperature impact analysis** on output quality